In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plts
from scipy.signal import find_peaks

from pmt.io import *
from pmt.io import load_files_streaming
import gc
import pandas as pd
from pmt.preprocessing import *
from pmt.config import *
from pmt.plotting import *
from pmt.selection import *



plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.60,
})


# Select Files


In [ ]:
channel="Channel 4"
data_dir = Path(globals().get("batch_data_dir", "PMT_Data"))

# file_name = "WA0089_750V_525MHz_led100" 
# file_name = "WA0089_775V_525MHz_led100" 
# file_name = "WA0089_800V_525MHz_led100"  
file_name = globals().get("batch_file_name", "WA0089_825V_525MHz_led100")
# file_name = "WA0089_850V_525MHz_led100"
# file_name = "WA0089_875V_525MHz_led100"
# file_name = "WA0089_900V_525MHz_led100"
# file_name = "WA0089_925V_525MHz_led100" 
# file_name = "WA0089_950V_525MHz_led100"
# file_name = "WA0089_975V_525MHz_led100" 
# file_name = "WA0089_1000V_525MHz_led100" 
# file_name = "WA0089_850V_20MHz_led100" 

# Set to a number for quick iteration, or None to use all matching files.
max_files = None

files = find_pmt_files(data_dir, file_name, max_files=max_files)
print(f"Loading {len(files)} files for {file_name} from {data_dir}")

# save
save_plots = globals().get("batch_save_plots", False)
save_dir = str(globals().get("batch_selection_output_dir", Path('plots/260706')))
file_nickname = file_name

fit_inputs = str(globals().get("batch_fit_inputs_dir", Path("fit_data")))
fit_inputs_path = Path(fit_inputs)
fit_inputs_path.mkdir(parents=True, exist_ok=True)

In [ ]:
chunk_size = 512 # files are read in chunks, it speeds up things
baseline_window_ns = (0, 20)

# Fixed, trigger-relative LED integration window: [36-20, 36+80] ns.
# Change led_time_ns if the independently measured LED timing changes.
led_time_ns = 36.0
pre_led_ns = 20.0
post_led_ns = 80.0
peak_snr_threshold = float(globals().get("batch_peak_snr_threshold", 5.0))
peak_prominence_snr = globals().get("batch_peak_prominence_snr", None)
peak_distance_samples = globals().get("batch_peak_distance_samples", None)
peak_width_samples = globals().get("batch_peak_width_samples", None)

# Load files

1. read files
2. remove saturated events
3. subtract baseline
4. remove envents that peak in the baseline window


returns:
- time_ns: array of time in ns 
- waveforms_mW: array of voltages in mV (each roe corresponds to a waveform)
- df: pandas dataframe containing useful information such as, peak time, amplitude, charge...

In [ ]:

# time_ns, waveforms_mV, df =  load_files_one_go(files, chunk_size, baseline_window_ns, channel)

time_ns, df_sel, waveforms_sample = load_files_streaming(
    files,
    channel=channel,
    chunk_size=512,
    baseline_window_ns=baseline_window_ns,
    led_time_ns=led_time_ns,
    pre_led_ns=pre_led_ns,
    post_led_ns=post_led_ns,
    peak_snr_threshold=peak_snr_threshold,
    peak_prominence_snr=peak_prominence_snr,
    peak_distance_samples=peak_distance_samples,
    peak_width_samples=peak_width_samples,
    keep_waveform_sample=1000,   # use 0 if you do not need waveform plotting
)

# The streaming loader keeps the first selected waveforms in dataframe order.
# Keep the matching dataframe rows for plots that use waveforms_sample.
df_waveforms = df_sel.iloc[:len(waveforms_sample)].copy()

requested_led_window_ns = (led_time_ns - pre_led_ns, led_time_ns + post_led_ns)
recorded_led_window_ns = (
    max(time_ns[0], requested_led_window_ns[0]),
    min(time_ns[-1], requested_led_window_ns[1]),
)
led_window_coverage = df_sel['charge_led_window_coverage'].iloc[0]
print(f"Requested LED charge window: {requested_led_window_ns} ns")
print(f"Recorded part of LED window: {recorded_led_window_ns} ns")
print(f"LED-window coverage: {100 * led_window_coverage:.1f}%")

# save dataframe to file: it speeds up the fit greatly (from several minutes to seconds)
df_sel.to_pickle(f"{fit_inputs}/{file_nickname}_df.pkl") 


In [ ]:
df_sel.columns

# Some plots

In [ ]:
n_to_plot = 500

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

add_plot_waveforms(axes[0], time_ns, waveforms_sample[:n_to_plot], title=f"Preprocessed Data - evts 0-{n_to_plot}")
axes[0].axvspan(*baseline_window_ns, color="tab:orange", alpha=0.25, label="baseline")
axes[0].legend(loc="lower right")

add_plot_waveforms(axes[1], time_ns, waveforms_sample[n_to_plot:2*n_to_plot], title=f"Preprocessed Data - evts {n_to_plot}-{2*n_to_plot}")
axes[1].axvspan(*baseline_window_ns, color="tab:orange", alpha=0.25, label="baseline")
axes[1].legend(loc="lower right")

fig.tight_layout()
save_plot(fig, save_plots, save_dir, file_nickname,"few_waveforms", Nevents=n_to_plot);

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.ravel()
plot_snr_distribution(axes[0], df_sel)
plot_mean_waveforms_vs_snr(axes[1], df_waveforms, waveforms_sample, time_ns, cuts=(2, 5, 8, 10, 13, 15, 20))
plot_charge_histograms_vs_snr(axes[2], df_sel, cuts=(2, 5, 8, 10, 13, 15, 20))
plot_snr_efficiency(axes[3], df_sel, cuts=np.arange(1, 25))
plot_waveforms_in_snr_range(axes[4], df_waveforms, waveforms_sample, time_ns, snr_min=13, snr_max=50, n_waveforms=50)
plot_snr_vs_amplitude(axes[5], df_sel)  
save_plot(fig, save_plots, save_dir, file_nickname, f"snr_repport",  Nevents=None);

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(15, 5))


rg = None #(-20, 10)
den=False

axes.hist(df_sel.charge_peak_window_mV_ns,  bins=100, range=rg, density=den, histtype='step', label="per-waveform peak window");
axes.hist(df_sel.charge_led_window_mV_ns, bins=100, range=rg, density=den, histtype='step', label="fixed LED window (-20, +80 ns)");
axes.hist(df_sel.charge_cfd10_window_mV_ns, bins=100, range=rg, density=den, histtype='step', label="rise time 10%");
axes.hist(df_sel.charge_cfd20_window_mV_ns, bins=100, range=rg, density=den, histtype='step', label="rise time 20%");
axes.hist(df_sel.charge_cfd30_window_mV_ns, bins=100, range=rg, density=den, histtype='step', label="rise time 30%");
axes.hist(df_sel.charge_cfd50_window_mV_ns, bins=100, range=rg, density=den, histtype='step', label="rise time 50%");
axes.hist(df_sel.area_mV_ns, bins=100, range=rg, density=den, histtype='step', label="full window");
axes.set_yscale('log');
axes.set_xlabel('Charge  [mV.ns]');
axes.set_ylabel('Counts');
axes.legend();

save_plot(fig, save_plots, save_dir, file_nickname, f"charge_all_methods",  Nevents=None);



# Baseline Study

In [ ]:
from scipy import stats
from scipy.stats import norm
from scipy.stats import skew, kurtosis
from scipy.signal import correlate
from scipy.signal import welch

In [ ]:
t, wfs, df = time_ns, waveforms_sample, df_sel

In [ ]:
mask = (t >= baseline_window_ns[0]) & (t <= baseline_window_ns[1])
baseline_region = wfs[:, mask]
fig, axes = plt.subplots(1, 1, figsize=(12, 5), sharex=True)
add_plot_waveforms(axes, t[mask], baseline_region[:5000], title='baseline region')
save_plot(fig, save_plots, save_dir, file_nickname, f"Baseline",  Nevents=5000);

In [ ]:
samples = baseline_region.ravel()
mu, sigma = norm.fit(samples)
sk = skew(samples)
kurt = kurtosis(samples)

stats_text = ( f"Skew = {sk:.3f}\n" f"Excess kurtosis = {kurt:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes = axes.ravel()

x = np.linspace(samples.min(), samples.max(), 500)
axes[0].hist(samples, bins=200, density=True, histtype='step', label='Baseline samples')
axes[0].plot(x, norm.pdf(x, mu, sigma), label=f'Gaussian fit\nμ={mu:.3f} mV\nσ={sigma:.3f} mV')
axes[0].text( -0.6, 1.0, stats_text, ha='right', va='top',
          bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray') )
axes[0].set_xlabel("Voltage (mV)")
axes[0].set_ylabel("Probability density")
axes[0].set_title("Fit to baseline samples")
axes[0].legend();

rng = np.random.default_rng(42)
idx = rng.choice(len(samples), size=10_000, replace=False)
stats.probplot(samples[idx], dist="norm", plot=plt)
axes[1].set_title("Gaussian Q-Q plot")
save_plot(fig, save_plots, save_dir, file_nickname, f"baseline_fit",  Nevents=None);





In [ ]:
corr = np.zeros(baseline_region.shape[1])
for wf in baseline_region:
    c = correlate(wf, wf, mode='full')
    c = c[c.size//2:]
    corr += c / c[0]          # normalize
corr /= len(baseline_region)
lags = np.arange(len(corr))


dt = np.median(np.diff(t))
print(f"Sampling step: {dt:.5f} ns")
fs = 1 / (dt * 1e-9)          # sampling frequency (Hz)
psds = []
for wf in baseline_region:
    f, Pxx = welch( wf, fs=fs, nperseg=len(wf), detrend='constant')
    psds.append(Pxx)
psd = np.mean(psds, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes = axes.ravel()

axes[0].plot(lags, corr)
axes[0].set_title("Baseline autocorrelation")
axes[0].set_xlabel("Lag (samples)")
axes[0].set_ylabel("Autocorrelation")
axes[0].grid(True);

axes[1].semilogy(f/1e6, psd)
axes[1].set_title("Power Spectral Density")
axes[1].set_xlabel("Frequency (MHz)")
axes[1].set_ylabel("PSD (mV²/Hz)")
axes[1].grid(True)

save_plot(fig, save_plots, save_dir, file_nickname, f"autocorrelation",  Nevents=None);

In [ ]:
C = np.cov(baseline_region.T)

fig, axes = plt.subplots(1, 1, figsize=(8, 6))

plt.imshow(C,
           origin='lower',
           cmap='RdBu_r',
           aspect='auto')
plt.colorbar(label="Covariance")
plt.title("Covariance matrix")
plt.xlabel("Sample")
plt.ylabel("Sample")

save_plot(fig, save_plots, save_dir, file_nickname, f"covariance_matrix",  Nevents=None);

# Further cuts?

In [ ]:
# Estimate the LED-synchronous peak time from high-SNR events. SNR adapts
# the pulse threshold to each waveform's measured baseline noise. If the
# expected delay is known independently, assign it directly instead.
timing_reference_snr = float(globals().get("batch_timing_reference_snr", 30.0))

timing_reference_mask = (
    (df_sel["snr"] >= timing_reference_snr)
    & (df_sel["n_peaks"] == 1)
    & np.isfinite(df_sel["peak_time_ns"])
)
timing_reference = df_sel.loc[timing_reference_mask, "peak_time_ns"]

if timing_reference.empty:
    raise ValueError("No clean events are available to estimate the LED peak time.")

timing_bin_width_ns = 1.0
timing_bins = np.arange(
    timing_reference.min(),
    timing_reference.max() + timing_bin_width_ns,
    timing_bin_width_ns,
)
timing_counts, timing_edges = np.histogram(timing_reference, bins=timing_bins)
peak_bin = np.argmax(timing_counts)
expected_peak_time_ns = 0.5 * (timing_edges[peak_bin] + timing_edges[peak_bin + 1])

# Tune this after inspecting the high-SNR timing distribution below.
peak_timing_tolerance_ns = float(globals().get("batch_peak_timing_tolerance_ns", 5.0))
allowed_peak_window_ns = (
    expected_peak_time_ns - peak_timing_tolerance_ns,
    expected_peak_time_ns + peak_timing_tolerance_ns,
)

led_timing_offset_ns = expected_peak_time_ns - led_time_ns
print(f"Configured LED integration time: {led_time_ns:.2f} ns")
print(f"Expected peak time: {expected_peak_time_ns:.2f} ns")
print(f"Peak - configured LED time: {led_timing_offset_ns:+.2f} ns")
print(f"Timing reference: SNR >= {timing_reference_snr:g}")
print(f"Allowed peak window: {allowed_peak_window_ns} ns")

# Learn loose pulse-shape ranges from clean LED-synchronous pulses. These
# central 98% intervals remove pathological shapes while retaining the
# natural SPE spread. Recompute them for every PMT voltage/run.
shape_columns = [
    "peak_width_ns",
    "rise_time_10_90_ns",
    "fall_time_90_10_ns",
]
shape_reference_mask = (
    timing_reference_mask
    & df_sel["peak_time_ns"].between(*allowed_peak_window_ns)
    & np.isfinite(df_sel[shape_columns]).all(axis=1)
)
shape_reference = df_sel.loc[shape_reference_mask, shape_columns]
min_shape_reference_pulses = 100
shape_cut_available = len(shape_reference) >= min_shape_reference_pulses
shape_cut_ranges = {}
if shape_cut_available:
    shape_quantiles = (0.01, 0.99)
    shape_cut_ranges = {
        column: tuple(shape_reference[column].quantile(shape_quantiles))
        for column in shape_columns
    }
else:
    print(
        f"Warning: only {len(shape_reference)} clean pulses are available; "
        f"at least {min_shape_reference_pulses} are required to derive stable "
        "pulse-shape ranges. The pulse-shape cut will be skipped."
    )
is_pulse_shaped = np.ones(len(df_sel), dtype=bool)
for column, limits in shape_cut_ranges.items():
    is_pulse_shaped &= df_sel[column].between(*limits).to_numpy()
    print(f"{column}: {limits[0]:.3g} to {limits[1]:.3g} ns")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

# Plot all populations with the same bin edges so their shapes and counts
# can be compared directly. timing_reference is the exact population used
# above to determine expected_peak_time_ns.
finite_peak_times = df_sel.loc[
    np.isfinite(df_sel["peak_time_ns"]), "peak_time_ns"
]
timing_plot_bins = np.linspace(
    finite_peak_times.min(), finite_peak_times.max(), 201
)
above_threshold_peak_times = df_sel.loc[
    (df_sel["snr"] >= timing_reference_snr)
    & np.isfinite(df_sel["peak_time_ns"]),
    "peak_time_ns",
]

ax.hist(
    finite_peak_times, bins=timing_plot_bins, histtype="step",
    label=f"All events (N={len(finite_peak_times):,})",
)
ax.hist(
    above_threshold_peak_times, bins=timing_plot_bins, histtype="step",
    label=f"SNR >= {timing_reference_snr:g} (N={len(above_threshold_peak_times):,})",
)
ax.hist(
    timing_reference, bins=timing_plot_bins, histtype="step", linewidth=2,
    label=(f"Timing reference: SNR >= {timing_reference_snr:g} "
           f"and one peak (N={len(timing_reference):,})"),
)
ax.axvline(
    expected_peak_time_ns, color="tab:red",
    label=f"Mode of timing reference: {expected_peak_time_ns:.1f} ns",
)
ax.axvspan(*allowed_peak_window_ns, color="tab:green", alpha=0.2, label="Accepted timing window")
ax.set_xlabel("Peak time [ns]")
ax.set_ylabel("Events")
ax.set_yscale("log")
ax.set_title("Populations used for the timing selection")
ax.legend()
fig.tight_layout();

In [ ]:

# Apply the single-peak, LED-timing, and learned pulse-shape requirements
# above several SNR thresholds. Events below threshold are preserved so the
# pedestal population needed by the occupancy fit is not removed.
cut_thresholds_snr = [float(x) for x in globals().get("batch_cut_thresholds_snr", [2, 5,8, 10, 15])]
include_shape_cut = bool(globals().get("batch_include_shape_cut", True))
is_single_peak = df_sel["n_peaks"] == 1
is_led_aligned = df_sel["peak_time_ns"].between(*allowed_peak_window_ns)

selection_results = {}
selection_cutflows = {}

for threshold_snr in cut_thresholds_snr:
    threshold_tag = f"snr{threshold_snr:g}"
    is_over_threshold = df_sel["snr"] >= threshold_snr
    is_at_or_below_threshold = ~is_over_threshold

    print(f"\n===== Cuts applied at SNR >= {threshold_snr:g} =====")
    cutflow = CutFlow(len(df_sel))
    cutflow.apply(
        f"SNR >= {threshold_snr:g}: single peak",
        is_at_or_below_threshold | is_single_peak,
    )
    cutflow.apply(
        f"SNR >= {threshold_snr:g}: LED peak timing",
        is_at_or_below_threshold | is_led_aligned,
    )
    if include_shape_cut and shape_cut_available:
        cutflow.apply(
            f"SNR >= {threshold_snr:g}: pulse shape",
            is_at_or_below_threshold | is_pulse_shaped,
        )
    cutflow.print()

    df_selected_snr = df_sel.loc[cutflow.mask].copy()
    selection_results[threshold_snr] = df_selected_snr
    selection_cutflows[threshold_snr] = cutflow

    selected_df_path = (
        fit_inputs_path
        / f"{file_nickname}_df_selected_{threshold_tag}.pkl"
    )
    df_selected_snr.to_pickle(selected_df_path)
    print(f"Saved {len(df_selected_snr):,} selected events to:")
    print(selected_df_path)

# Keep the original variable and filename as aliases for one selected SNR.
# Prefer SNR >= 20 when it was generated, otherwise use the first configured threshold.
default_selection_snr = 20.0 if 20.0 in selection_results else cut_thresholds_snr[0]
df_selected = selection_results[default_selection_snr]
df_selected.to_pickle(fit_inputs_path / f"{file_nickname}_df_selected.pkl")
cutflow = selection_cutflows[default_selection_snr]

# For waveform plots below, keep the default selection as the default.
if (
    "waveforms_sample" in globals()
    and waveforms_sample is not None
    and "df_waveforms" in globals()
):
    sample_cut_mask = cutflow.mask[:len(waveforms_sample)]
    df_waveforms_selected = df_waveforms.loc[sample_cut_mask].copy()
    waveforms_selected = waveforms_sample[sample_cut_mask]
else:
    print("Waveform sample is not loaded; skipping waveform selection.")


In [ ]:
n_to_plot = 500

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

add_plot_waveforms(axes[0], time_ns, waveforms_sample[:n_to_plot], title="Before cuts")
axes[0].axvspan(*baseline_window_ns, color="tab:orange", alpha=0.25, label="baseline")
axes[0].legend(loc="lower right")

add_plot_waveforms(axes[1], time_ns, waveforms_selected[:n_to_plot], title="After cuts")
axes[1].axvspan(*baseline_window_ns, color="tab:orange", alpha=0.25, label="baseline")
axes[1].axvspan(*allowed_peak_window_ns, color="tab:green", alpha=0.20, label="allowed peak time")
axes[1].legend(loc="lower right")

fig.tight_layout();
